### LIBRARY IMPORTS

In [2]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
import copy
import warnings

from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_manager import DataManager
from src.processor import Processor
from src.cnn_regressor import CNNRegressor
from src.gb_classifier import GBClassifier

warnings.filterwarnings("ignore")

KeyboardInterrupt: 

### CONFIGURATION

In [1]:
data_manager = DataManager()

datasets_config, modeling_config = data_manager.load_config()

active_dataset = modeling_config["main"]["active_dataset"]
active_dataset_config = datasets_config[active_dataset]

problem_type = active_dataset_config["problem_type"]

gradient_boosting_config = modeling_config["gradient_boosting"]

weak_learner_key = gradient_boosting_config["weak_learner_key"]
weak_learner_config = modeling_config[weak_learner_key]

processor = Processor(**active_dataset_config)

train, test = data_manager.load_image_data(active_dataset)

X, y = processor.split_features_target(train)
X_test, y_test = processor.split_features_target(test)

y, y_test = y.reshape(-1, 1), y_test.reshape(-1, 1)

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

NameError: name 'DataManager' is not defined

In [35]:
X_train.shape

(48000, 1, 28, 28)

### EVALUATION FUNCTION

In [36]:
def fit_and_evaluate(model, flatten=False, **extra_params):
    xt, xv, xte = X_train, X_valid, X_test

    if flatten:
        xt = xt.reshape(xt.shape[0], -1)
        xv = xv.reshape(xv.shape[0], -1)
        xte = xte.reshape(xte.shape[0], -1)

    model.fit(xt, y_train, **extra_params)

    valid_prob_preds = model.predict_proba(xv)
    valid_class_preds = np.argmax(valid_prob_preds, axis=1)

    valid_log_loss = log_loss(y_valid, valid_prob_preds, labels=np.unique(y_train))
    valid_accuracy = accuracy_score(y_valid, valid_class_preds)

    test_prob_preds = model.predict_proba(xte)
    test_class_preds = np.argmax(test_prob_preds, axis=1)

    test_log_loss = log_loss(y_test, test_prob_preds, labels=np.unique(y_train))
    test_accuracy = accuracy_score(y_test, test_class_preds)
    
    return valid_log_loss, valid_accuracy, test_log_loss, test_accuracy

### LOGISTIC REGRESSION

In [40]:
%%time

logistic = LogisticRegression(C=1.0, random_state=42)
logistic_valid_log_loss, logistic_valid_accuracy, logistic_test_log_loss, logistic_test_accuracy = fit_and_evaluate(logistic, flatten=True)

CPU times: total: 1min 11s
Wall time: 9.17 s


In [44]:
print(f"Logistic Valid Log Loss: {logistic_valid_log_loss:.6f}")
print(f"Logistic Valid Accuracy: {logistic_valid_accuracy:.6f}")
print("-" * 50)
print(f"Logistic Test Log Loss:  {logistic_test_log_loss:.6f}")
print(f"Logistic Test Accuracy:  {logistic_test_accuracy:.6f}")

Logistic Valid Log Loss: 0.283924
Logistic Valid Accuracy: 0.923417
--------------------------------------------------
Logistic Test Log Loss:  0.272249
Logistic Test Accuracy:  0.925400


### DECISION TREE

In [43]:
%%time

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_valid_log_loss, dt_valid_accuracy, dt_test_log_loss, dt_test_accuracy = fit_and_evaluate(dt, flatten=True)

CPU times: total: 3.84 s
Wall time: 3.89 s


In [45]:
print(f"Decision Tree Valid Log Loss: {dt_valid_log_loss:.4f}")
print(f"Decision Tree Valid Accuracy: {dt_valid_accuracy:.4f}")
print("-" * 50)
print(f"Decision Tree Test Log Loss:  {dt_test_log_loss:.4f}")
print(f"Decision Tree Test Accuracy:  {dt_test_accuracy:.4f}")

Decision Tree Valid Log Loss: 1.1510
Decision Tree Valid Accuracy: 0.6579
--------------------------------------------------
Decision Tree Test Log Loss:  1.1249
Decision Tree Test Accuracy:  0.6631


### CONVOLUTIONAL NEURAL NETWORKS

Big CNN total parameters: 46,730

Small CNN total parameters: 3,154

In [ ]:
class CNN(CNNRegressor):
    def __init__(self, **hyperparameters):
        super().__init__(**hyperparameters)

    def fit(self, X, y):
        
        in_channels = X.shape[1]
        output_size = int(np.max(y)) + 1

        image_size = X.shape[-1]

        conv1_out = image_size - self.kernel_size + 1
        pool1_out = conv1_out // self.pool_size

        conv2_out = pool1_out - self.kernel_size + 1
        pool2_out = conv2_out // self.pool_size

        linear_input = self.channels[1] * pool2_out * pool2_out
        
        self._get_network(in_channels, linear_input, output_size)

        loader = self._prepare_loader(X, y)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters(), self.learning_rate)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, 
            max_lr=self.learning_rate, 
            steps_per_epoch=len(loader), 
            epochs=self.epochs
        )

        self.train()
        for _ in range(self.epochs):
            for batch_X, batch_y in loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                
                loss = criterion(preds, batch_y.long().view(-1))
                loss.backward()
                optimizer.step()
                scheduler.step()

In [ ]:
big_cnn = CNN(epochs=5, learning_rate=0.01, channels=[16, 32], kernel_size=5, pool_size=2, hidden_size=64, batch_size=256)
big_cnn.fit(X_train, y_train)

raw_preds = big_cnn.predict(X_test) 

if len(raw_preds.shape) > 1 and raw_preds.shape[1] > 1:
    preds = np.argmax(raw_preds, axis=1)
else:
    preds = raw_preds.flatten().astype(int)

score = accuracy_score(y_test, preds)
print(f"Test Accuracy: {score:.4f}")

--------------------------------------------------
Test Accuracy: 0.9933


In [24]:
small_cnn = CNN(epochs=5, learning_rate=0.01, channels=[4, 8], kernel_size=5, pool_size=2, hidden_size=16, batch_size=256)
small_cnn.fit(X_train, y_train)


raw_preds = small_cnn.predict(X_test) 

if len(raw_preds.shape) > 1 and raw_preds.shape[1] > 1:
    preds = np.argmax(raw_preds, axis=1)
else:
    preds = raw_preds.flatten().astype(int)

score = accuracy_score(y_test, preds)
print(f"Test Accuracy: {score:.4f}")

Test Accuracy: 0.9812
